# Previsão de Séries Temporais — Atividade Aula 02
## Escolha de Domínio e Dados para o Projeto da Disciplina

**Aluno:** Gabriel Floriano
**Disciplina:** Previsão de Séries Temporais — PUCRS
**Profª:** Katherine Bianchini Esper de Vargas

Esta atividade cobre as **duas primeiras etapas do CRISP-DM**:

| Etapa | Situação nesta entrega |
|---|---|
| 1. Compreensão do negócio | ✅ Passo 2 |
| 2. Compreensão dos dados | ✅ Passo 3 (perfil + exploração visual) |
| 3. Preparação dos dados | — próximas aulas |
| 4. Modelagem | — próximas aulas |
| 5. Avaliação | — próximas aulas |
| 6. Comunicação | — próximas aulas |

> A análise do Passo 3 é **exploratória e visual**, sem testes estatísticos formais.
> O objetivo é formar hipóteses iniciais sobre **tendência** e **sazonalidade**.

---
# Passo 1 — Domínio escolhido e justificativa

## Domínio: **Segurança Pública — ocorrências criminais no Brasil, por estado**

**Por que este domínio?**

1. **É um fenômeno genuinamente temporal.** Ocorrências criminais são contabilizadas em
   janelas fixas (mês a mês, por unidade da federação), acumulam memória — o nível de um mês
   é fortemente informado pelo mês anterior — e reagem com defasagem a intervenções de política
   pública. É exatamente o tipo de processo que uma série temporal descreve bem.

2. **Tem os três componentes clássicos ao mesmo tempo.** A literatura de criminologia aponta
   *tendência* de longo prazo (efeito de mudanças demográficas, de política penal e de
   policiamento), *sazonalidade* intra-anual (calor, férias escolares, festas de fim de ano,
   13º salário e maior circulação de pessoas e dinheiro) e *choques* pontuais bem
   identificáveis (a pandemia de COVID-19 em 2020, por exemplo, produziu uma quebra abrupta
   na mobilidade e, com ela, nos crimes patrimoniais). Isso torna o dataset didaticamente rico
   para decomposição, modelos sazonais e detecção de quebras estruturais.

3. **É um painel, não uma única série.** Como os dados são desagregados por UF e por tipo de
   crime, é possível trabalhar de várias formas ao longo do semestre: uma série agregada Brasil,
   séries por estado, séries por tipo de crime, ou previsão hierárquica com reconciliação
   (o total do Brasil deve bater com a soma das UFs).

4. **Relevância social direta e mensurável.** Diferente de um exercício puramente técnico, aqui
   um erro de previsão tem custo concreto: efetivo policial mal alocado, orçamento mal
   dimensionado, política preventiva aplicada no mês errado.

5. **Dado público, gratuito e com histórico longo o suficiente** para estimar sazonalidade anual
   (é preciso pelo menos 2–3 ciclos completos; verificaremos isso no Passo 3).

---
# Passo 2 — Compreensão do negócio

## 2.1 Qual é a problemática concreta?

A gestão da segurança pública no Brasil é, na prática, **reativa e retrospectiva**. As Secretarias
Estaduais de Segurança Pública (SSPs) publicam boletins com o número de ocorrências do mês
anterior e a comparação com o mesmo mês do ano passado — a chamada "variação YoY". Esse
indicador, porém, responde apenas *o que já aconteceu*, e responde mal: uma alta de 8% em
dezembro pode ser simplesmente a sazonalidade normal de dezembro, e não uma deterioração real
da segurança. Sem separar tendência, sazonalidade e ruído, o gestor não sabe distinguir
**variação esperada** de **sinal de alerta**.

Isso se desdobra em três problemas operacionais:

- **Alocação de efetivo em cima da hora.** O dimensionamento de escalas, horas extras e
  operações de reforço é decidido com pouca antecedência, tipicamente reagindo ao mês que já
  piorou — quando o custo de resposta já é maior e o dano já ocorreu.
- **Orçamento anual construído sobre a média histórica.** A previsão orçamentária das SSPs e a
  definição de metas de redução tendem a usar médias simples ou a repetição do ano anterior,
  ignorando tendência e o efeito de choques recentes.
- **Avaliação de política sem contrafactual.** Quando um programa de prevenção é implantado,
  a avaliação usual compara "antes e depois". Sem um contrafactual — o que teria acontecido
  na ausência do programa — não é possível atribuir a queda observada à política em vez de a
  uma tendência que já estava em curso.

## 2.2 Para quem essa informação importa?

| Ator | Uso da informação |
|---|---|
| **SSPs estaduais / Polícia Militar e Civil** | Dimensionamento de efetivo, escalas, operações sazonais |
| **MJSP / SENASP** | Repasse de recursos do Fundo Nacional de Segurança Pública, comparação entre UFs |
| **Prefeituras e guardas municipais** | Planejamento de iluminação, videomonitoramento, patrulhamento |
| **Setor privado (seguradoras, varejo, logística)** | Precificação de risco, definição de rotas, segurança patrimonial |
| **Sociedade civil, imprensa e academia** | Fiscalização de metas públicas, pesquisa aplicada |

## 2.3 Que decisão real poderia ser apoiada por uma previsão?

1. **Dimensionar efetivo e horas extras com 1 a 3 meses de antecedência.** Uma previsão mensal
   por UF permite reforçar preventivamente os meses de pico projetado em vez de reagir depois.
2. **Definir metas de redução realistas e auditáveis.** A meta deixa de ser um número político
   ("reduzir 10%") e passa a ser medida contra a previsão-base: reduzir 10% *abaixo do que a
   série projetava*.
3. **Planejar o orçamento anual** das SSPs a partir do volume projetado de ocorrências, não da
   média histórica.
4. **Disparar alertas precoces.** Um mês que cai fora do intervalo de previsão é um sinal
   estatístico de mudança de regime — não apenas flutuação — e justifica investigação imediata.
5. **Avaliar impacto de políticas com contrafactual.** Congela-se o modelo antes da intervenção
   e compara-se o realizado com o previsto; a diferença é a estimativa do efeito.

## 2.4 De que forma isso agrega valor prático?

O ganho central é **separar o que é padrão do que é notícia**. Uma série decomposta responde a
perguntas que o boletim mensal não responde:

- *"Dezembro subiu 12% — isso é ruim?"* → Se a sazonalidade histórica de dezembro é +15%,
  o mês na verdade veio **melhor** que o esperado.
- *"A queda dos últimos 6 meses é a política nova funcionando?"* → Só se ela exceder a
  tendência que já vinha caindo antes da política.
- *"Onde alocar a próxima viatura?"* → Na UF cuja **tendência projetada** cresce, não
  necessariamente na de maior volume absoluto (que costuma ser apenas a mais populosa).

Antecipação também converte custo reativo em custo planejado: reforço programado é mais barato
que hora extra emergencial, e prevenção no mês certo evita o dano em vez de remediá-lo.

---
# Passo 3 — Compreensão dos dados

## 3.1 Dataset identificado

| Item | Descrição |
|---|---|
| **Nome** | Crimes no Brasil por Estado |
| **Link direto** | https://www.kaggle.com/datasets/vianags/crimes-no-brasil-por-estado |
| **Plataforma** | Kaggle (acesso gratuito, download via API `kagglehub`) |
| **Origem primária** | Estatísticas de segurança pública consolidadas a partir das Secretarias Estaduais de Segurança Pública (SSPs), publicadas pelo Ministério da Justiça e Segurança Pública / SENASP |
| **Formato** | CSV (tabular, leitura direta em `pandas`) |
| **Granularidade** | Ocorrências por **UF**, por **tipo de crime**, por **período** |

> ⚠️ Os itens **variável observada**, **frequência**, **período coberto** e **completude** são
> confirmados empiricamente pelas células abaixo, e não assumidos — o perfil impresso pelo
> notebook é a fonte da verdade para preencher o relatório.

## 3.2 Checklist de avaliação do dataset

| Critério | Como será verificado |
|---|---|
| A série está organizada no tempo? | Detecção e parsing da coluna temporal (§ 3.4) |
| Qual a frequência das observações? | Inferida pela mediana do intervalo entre datas (§ 3.5) |
| Qual o período histórico disponível? | Data mínima e máxima observadas (§ 3.5) |
| Os dados parecem completos? | Mapa de cobertura UF × período e contagem de faltantes (§ 3.6) |
| O formato é acessível e gratuito? | CSV público via `kagglehub` (§ 3.3) |

---
## 3.3 Aquisição dos dados e inventário dos arquivos

In [ ]:
# Instalação (execute apenas se necessário no seu ambiente)
# !pip install kagglehub pandas matplotlib statsmodels openpyxl

In [ ]:
import re
import warnings
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FuncFormatter

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 170)

# =====================================================================
# Identidade visual dos gráficos
# Paleta categórica de ordem fixa (nunca ciclar, nunca reordenar por
# ranking: a cor segue a entidade). Rampa sequencial de um único matiz
# para magnitude (heatmaps). Tinta recessiva para grade e eixos.
# =====================================================================
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
          "#e87ba4", "#008300", "#4a3aa7", "#e34948"]

SURFACE = "#fcfcfb"   # fundo do gráfico
INK     = "#0b0b0b"   # tinta primária (títulos)
INK_2   = "#52514e"   # tinta secundária (rótulos)
MUTED   = "#898781"   # eixos / ticks
GRID    = "#e1e0d9"   # grade fina
AXIS    = "#c3c2b7"   # linha de base
POS     = "#e34948"   # polaridade: aumento de crimes
NEG     = "#2a78d6"   # polaridade: queda de crimes

BLUE_RAMP = ["#cde2fb", "#b7d3f6", "#9ec5f4", "#86b6ef", "#6da7ec",
             "#5598e7", "#3987e5", "#2a78d6", "#256abf", "#1c5cab",
             "#184f95", "#104281", "#0d366b"]
CMAP_SEQ = LinearSegmentedColormap.from_list("seq_blue", BLUE_RAMP)

mpl.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE, "savefig.bbox": "tight",
    "font.family": ["DejaVu Sans"], "font.size": 10,
    "axes.titlesize": 12.5, "axes.titleweight": "semibold",
    "axes.titlecolor": INK, "axes.titlelocation": "left", "axes.titlepad": 16,
    "axes.labelcolor": INK_2, "axes.labelsize": 10,
    "axes.edgecolor": AXIS, "axes.linewidth": 0.8,
    "axes.grid": True, "axes.grid.axis": "y",
    "grid.color": GRID, "grid.linewidth": 0.8,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "xtick.labelcolor": INK_2, "ytick.labelcolor": INK_2,
    "xtick.labelsize": 9.5, "ytick.labelsize": 9.5,
    "legend.frameon": False, "legend.fontsize": 9.5,
    "lines.linewidth": 2.0, "lines.markersize": 5,
    "figure.dpi": 110,
})

def limpar(ax, eixo_grade="y"):
    # Remove moldura, mantém apenas a grade do eixo relevante.
    for lado in ("top", "right"):
        ax.spines[lado].set_visible(False)
    ax.spines["left"].set_color(AXIS)
    ax.spines["bottom"].set_color(AXIS)
    ax.set_axisbelow(True)
    ax.grid(axis="x", visible=(eixo_grade in ("x", "both")), color=GRID, linewidth=0.8)
    ax.grid(axis="y", visible=(eixo_grade in ("y", "both")), color=GRID, linewidth=0.8)
    return ax

def titular(ax, texto, subtitulo=None):
    ax.set_title(texto, color=INK, loc="left", pad=22 if subtitulo else 12)
    if subtitulo:
        ax.text(0.0, 1.015, subtitulo, transform=ax.transAxes,
                color=INK_2, fontsize=9.5, va="bottom", ha="left")
    return ax

def milhar(x, _pos=None):
    return f"{x:,.0f}".replace(",", ".")

FMT_MILHAR = FuncFormatter(milhar)

print("Ambiente configurado.")

In [ ]:
import kagglehub

CAMINHO = Path(kagglehub.dataset_download("vianags/crimes-no-brasil-por-estado"))
print("Path to dataset files:", CAMINHO)
print()

ARQUIVOS = sorted(p for p in CAMINHO.rglob("*") if p.is_file())
print(f"{len(ARQUIVOS)} arquivo(s) encontrado(s):")
for p in ARQUIVOS:
    print(f"  - {p.relative_to(CAMINHO)}  ({p.stat().st_size/1024:,.1f} KB)")

In [ ]:
def ler_arquivo(p):
    # Leitura tolerante: testa separador e encoding usuais em dados brasileiros.
    suf = p.suffix.lower()
    if suf in (".csv", ".txt", ".tsv"):
        for sep in (None, ";", ",", "\t", "|"):
            for enc in ("utf-8", "utf-8-sig", "latin-1"):
                try:
                    d = pd.read_csv(p, sep=sep, encoding=enc, engine="python")
                    if d.shape[1] > 1 and len(d) > 0:
                        return d
                except Exception:
                    continue
        return None
    if suf in (".xlsx", ".xls"):
        try:
            return pd.read_excel(p)
        except Exception:
            return None
    if suf == ".json":
        try:
            return pd.read_json(p)
        except Exception:
            return None
    return None

TABELAS = {}
for p in ARQUIVOS:
    d = ler_arquivo(p)
    if d is not None and len(d):
        TABELAS[p.name] = d
        print(f"OK  {p.name:<55} {d.shape[0]:>8,} linhas x {d.shape[1]:>3} colunas")

assert TABELAS, "Nenhuma tabela pôde ser lida — verifique o conteúdo do diretório baixado."

# Trabalhamos com a maior tabela; ajuste aqui se preferir outra.
NOME_TABELA = max(TABELAS, key=lambda k: TABELAS[k].size)
bruto = TABELAS[NOME_TABELA].copy()
print()
print(f"Tabela selecionada: {NOME_TABELA}  -> {bruto.shape[0]:,} linhas x {bruto.shape[1]} colunas")

---
## 3.4 Inspeção do schema e padronização

A célula abaixo imprime o schema **real** do arquivo. As colunas são então detectadas
automaticamente (data / UF / tipo de crime / quantidade). Se a detecção errar, basta preencher o
dicionário `MAPEAMENTO_MANUAL` com os nomes originais das colunas e reexecutar.

In [ ]:
print("=" * 78)
print("SCHEMA BRUTO")
print("=" * 78)
print(bruto.dtypes.to_frame("dtype").assign(
    nao_nulos=bruto.notna().sum(),
    nulos=bruto.isna().sum(),
    distintos=bruto.nunique(dropna=True),
))
print()
print("Primeiras linhas:")
display(bruto.head(10))
print()
print("Amostra de valores por coluna:")
for c in bruto.columns:
    amostra = bruto[c].dropna().unique()[:8]
    print(f"  {str(c)[:32]:<34} -> {list(amostra)}")

In [ ]:
# ---------------------------------------------------------------------
# Detecção automática de papéis das colunas
# ---------------------------------------------------------------------
MAPEAMENTO_MANUAL = {
    "data":  None,   # ex.: "mes_ano" — nome ORIGINAL da coluna, ou None para automático
    "ano":   None,
    "mes":   None,
    "uf":    None,
    "crime": None,
    "qtd":   None,
}

def normalizar(txt):
    t = unicodedata.normalize("NFKD", str(txt).strip().lower())
    t = t.encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]+", "_", t).strip("_")

UFS = ["AC","AL","AP","AM","BA","CE","DF","ES","GO","MA","MT","MS","MG","PA","PB",
       "PR","PE","PI","RJ","RN","RS","RO","RR","SC","SP","SE","TO"]

NOME_UF = {
    "acre":"AC","alagoas":"AL","amapa":"AP","amazonas":"AM","bahia":"BA","ceara":"CE",
    "distrito_federal":"DF","espirito_santo":"ES","goias":"GO","maranhao":"MA",
    "mato_grosso":"MT","mato_grosso_do_sul":"MS","minas_gerais":"MG","para":"PA",
    "paraiba":"PB","parana":"PR","pernambuco":"PE","piaui":"PI","rio_de_janeiro":"RJ",
    "rio_grande_do_norte":"RN","rio_grande_do_sul":"RS","rondonia":"RO","roraima":"RR",
    "santa_catarina":"SC","sao_paulo":"SP","sergipe":"SE","tocantins":"TO",
}

MES_PT = {"jan":1,"fev":2,"mar":3,"abr":4,"mai":5,"jun":6,
          "jul":7,"ago":8,"set":9,"out":10,"nov":11,"dez":12}

def para_uf(serie):
    s = serie.astype(str).str.strip()
    sigla = s.str.upper().where(s.str.len() == 2)
    nome = s.map(lambda v: NOME_UF.get(normalizar(v)))
    return sigla.where(sigla.isin(UFS)).fillna(nome)

def score_uf(serie):
    conv = para_uf(serie)
    return conv.notna().mean()

def tentar_datas(serie):
    # Devolve (datetime64 Series, rótulo do formato) ou (None, None).
    s = serie.astype(str).str.strip().str.lower()

    # 1) mês abreviado em português: "jan/15", "jan-2015", "jan 2015"
    ext = s.str.extract(r"^([a-z]{3})[a-z]*[\./\- ]+(\d{2,4})$")
    if ext[0].notna().mean() > 0.8 and ext[0].str[:3].isin(MES_PT).mean() > 0.8:
        mes = ext[0].str[:3].map(MES_PT)
        ano = pd.to_numeric(ext[1], errors="coerce")
        ano = ano.where(ano > 100, ano + 2000)
        d = pd.to_datetime(dict(year=ano, month=mes, day=1), errors="coerce")
        if d.notna().mean() > 0.8:
            return d, "mes_pt/ano"

    # 2) apenas o ano
    num = pd.to_numeric(s, errors="coerce")
    if num.notna().mean() > 0.9 and num.dropna().between(1970, 2100).mean() > 0.9:
        d = pd.to_datetime(num.astype("Int64").astype(str), format="%Y", errors="coerce")
        if d.notna().mean() > 0.8:
            return d, "ano"

    # 3) formatos de data reconhecidos pelo pandas
    for kwargs in ({"dayfirst": True}, {"dayfirst": False}):
        d = pd.to_datetime(serie, errors="coerce", **kwargs)
        if d.notna().mean() > 0.8:
            return d, "data"

    return None, None

def score_numerico(serie):
    # Aceita tanto notação americana (1234.5) quanto brasileira (1.234,5):
    # converte das duas formas e fica com a que aproveita mais linhas.
    if pd.api.types.is_numeric_dtype(serie):
        s = pd.to_numeric(serie, errors="coerce").astype(float)
        return s.notna().mean(), s
    txt = serie.astype(str).str.strip()
    americano = pd.to_numeric(txt, errors="coerce")
    brasileiro = pd.to_numeric(
        txt.str.replace(r"\.", "", regex=True).str.replace(",", ".", regex=False),
        errors="coerce")
    s = brasileiro if brasileiro.notna().mean() > americano.notna().mean() else americano
    return s.notna().mean(), s.astype(float)

# --- mapa nome normalizado -> nome original
COLS = {normalizar(c): c for c in bruto.columns}
usadas = set()

def escolher(chave, palavras, validador=None):
    manual = MAPEAMENTO_MANUAL.get(chave)
    if manual is not None:
        return manual
    candidatos = [orig for norm, orig in COLS.items()
                  if orig not in usadas and any(p in norm for p in palavras)]
    if validador is not None:
        candidatos = [c for c in candidatos if validador(bruto[c])]
    return candidatos[0] if candidatos else None

# UF -----------------------------------------------------------------
col_uf = MAPEAMENTO_MANUAL["uf"]
if col_uf is None:
    ranking = sorted(((score_uf(bruto[c]), c) for c in bruto.columns),
                     key=lambda t: -t[0])
    col_uf = ranking[0][1] if ranking and ranking[0][0] > 0.8 else None
if col_uf: usadas.add(col_uf)

# DATA ---------------------------------------------------------------
col_data, fmt_data, datas = MAPEAMENTO_MANUAL["data"], None, None
alvos = [col_data] if col_data else [c for c in bruto.columns if c not in usadas]
for c in alvos:
    d, f = tentar_datas(bruto[c])
    if d is not None:
        col_data, fmt_data, datas = c, f, d
        break

# fallback: colunas separadas de ano e mês
col_ano = MAPEAMENTO_MANUAL["ano"] or escolher("ano", ["ano", "year"])
col_mes = MAPEAMENTO_MANUAL["mes"] or escolher("mes", ["mes", "month"])
if datas is None and col_ano:
    ano = pd.to_numeric(bruto[col_ano], errors="coerce")
    if col_mes is not None:
        m = bruto[col_mes]
        mes = pd.to_numeric(m, errors="coerce")
        if mes.isna().mean() > 0.5:
            mes = m.astype(str).str.strip().str.lower().str[:3].map(MES_PT)
    else:
        mes = pd.Series(1, index=bruto.index)
    datas = pd.to_datetime(dict(year=ano, month=mes.fillna(1), day=1), errors="coerce")
    col_data, fmt_data = f"{col_ano}+{col_mes}", "ano+mes"
if col_data: usadas.update({col_data, col_ano, col_mes} - {None})

# QUANTIDADE ---------------------------------------------------------
col_qtd = MAPEAMENTO_MANUAL["qtd"]
if col_qtd is None:
    palavras = ["ocorrenc", "quantidade", "qtd", "total", "valor", "vitima", "registro", "casos", "n_"]
    cand = [c for c in bruto.columns if c not in usadas and any(p in normalizar(c) for p in palavras)]
    if not cand:
        cand = [c for c in bruto.columns if c not in usadas and score_numerico(bruto[c])[0] > 0.9]
    if cand:
        col_qtd = max(cand, key=lambda c: score_numerico(bruto[c])[1].fillna(0).sum())
if col_qtd: usadas.add(col_qtd)

# TIPO DE CRIME ------------------------------------------------------
col_crime = MAPEAMENTO_MANUAL["crime"]
if col_crime is None:
    cand = [c for c in bruto.columns
            if c not in usadas and bruto[c].dtype == object and 1 < bruto[c].nunique() <= 80]
    prefer = [c for c in cand if any(p in normalizar(c)
              for p in ["crime", "tipo", "delito", "ocorrenc", "natureza", "categoria"])]
    col_crime = (prefer or cand or [None])[0]

print("Papéis detectados")
print("-" * 50)
for papel, col in [("data", col_data), ("uf", col_uf), ("crime", col_crime), ("quantidade", col_qtd)]:
    print(f"  {papel:<12} -> {col}   {'(formato: ' + str(fmt_data) + ')' if papel == 'data' else ''}")
print()
print("Se algum papel estiver errado ou vazio, preencha MAPEAMENTO_MANUAL acima e reexecute.")

In [ ]:
# ---------------------------------------------------------------------
# Construção da tabela longa padronizada: data | uf | crime | qtd
# ---------------------------------------------------------------------
assert col_data is not None and col_qtd is not None, \
    "Data ou quantidade não detectadas — preencha MAPEAMENTO_MANUAL."

df = pd.DataFrame({"data": pd.to_datetime(datas)})
df["uf"]    = para_uf(bruto[col_uf]) if col_uf else "BR"
df["crime"] = bruto[col_crime].astype(str).str.strip() if col_crime else "Total de ocorrências"
df["qtd"]   = score_numerico(bruto[col_qtd])[1]

# Se existir uma coluna indicando se o valor é absoluto ou taxa, ficamos com o absoluto.
col_formato = next((c for c in bruto.columns if normalizar(c) in ("formato", "unidade", "medida")), None)
if col_formato is not None:
    marca = bruto[col_formato].astype(str).str.lower().str.startswith(("abs", "num", "ocor"))
    if 0 < marca.mean() < 1:
        print(f"Coluna '{col_formato}' encontrada — mantendo apenas registros absolutos "
              f"({marca.sum():,} de {len(marca):,}).")
        df = df[marca.values]

antes = len(df)
df = df.dropna(subset=["data", "qtd"]).reset_index(drop=True)
df["qtd"] = df["qtd"].astype(float)
df["ano"] = df["data"].dt.year
df["mes"] = df["data"].dt.month

print(f"Linhas descartadas por data/quantidade ausente: {antes - len(df):,}")
print(f"Tabela padronizada: {len(df):,} linhas")
display(df.head(10))

---
## 3.5 Perfil temporal: frequência, período e volume

In [ ]:
periodos = pd.Index(sorted(df["data"].unique()))
delta = pd.Series(periodos).diff().dt.days.median()
FREQ = ("diária" if delta <= 2 else "semanal" if delta <= 9 else
        "mensal" if delta <= 45 else "trimestral" if delta <= 135 else "anual")
FREQ_PANDAS = {"diária": "D", "semanal": "W", "mensal": "MS",
               "trimestral": "QS", "anual": "YS"}[FREQ]

n_ufs = df["uf"].nunique()
n_crimes = df["crime"].nunique()
anos_cobertos = (df["data"].max() - df["data"].min()).days / 365.25

perfil = pd.Series({
    "Variável observada":       f"Nº de ocorrências ({col_qtd})",
    "Frequência":               FREQ,
    "Intervalo mediano (dias)": f"{delta:.0f}",
    "Início da série":          df["data"].min().strftime("%Y-%m-%d"),
    "Fim da série":             df["data"].max().strftime("%Y-%m-%d"),
    "Períodos distintos":       f"{len(periodos):,}",
    "Anos cobertos":            f"{anos_cobertos:.1f}",
    "Ciclos anuais completos":  f"{int(anos_cobertos)}",
    "Unidades federativas":     f"{n_ufs}",
    "Tipos de crime":           f"{n_crimes}",
    "Linhas":                   f"{len(df):,}",
    "Total de ocorrências":     f"{df['qtd'].sum():,.0f}".replace(",", "."),
}, name="valor").to_frame()

print("PERFIL DO DATASET")
display(perfil)

print()
print(f"Combinações esperadas (períodos x UFs x crimes): "
      f"{len(periodos) * n_ufs * n_crimes:,}")
print(f"Combinações observadas:                          {len(df):,}")
print(f"Cobertura da grade completa:                     "
      f"{100 * len(df) / (len(periodos) * n_ufs * n_crimes):.1f}%")

if int(anos_cobertos) >= 3:
    print()
    print(f"✅ {int(anos_cobertos)} ciclos anuais — suficiente para estimar sazonalidade anual.")
else:
    print()
    print(f"⚠️  Apenas {anos_cobertos:.1f} anos — sazonalidade anual será mal identificada.")

In [ ]:
print("Tipos de crime presentes:")
for i, (c, v) in enumerate(df.groupby("crime")["qtd"].sum().sort_values(ascending=False).items(), 1):
    print(f"  {i:>2}. {str(c):<48} {milhar(v):>16}")
print()
print(f"UFs presentes ({n_ufs}): {', '.join(sorted(df['uf'].dropna().unique()))}")
faltando = sorted(set(UFS) - set(df["uf"].dropna().unique()))
print(f"UFs ausentes ({len(faltando)}): {', '.join(faltando) if faltando else 'nenhuma'}")

---
## 3.6 Características e qualidade dos dados

Os gráficos desta seção descrevem **o dataset em si** — onde há lacunas, como os valores se
distribuem, e quanto cada UF e cada tipo de crime pesa no total. Eles orientam a etapa de
preparação dos dados (Etapa 3 do CRISP-DM).

In [ ]:
# ---- Gráfico 1: mapa de cobertura temporal (UF x período) ------------
cobertura = (df.assign(presente=1)
               .pivot_table(index="uf", columns="data", values="presente",
                            aggfunc="max", fill_value=0)
               .sort_index())

fig, ax = plt.subplots(figsize=(13, 7))
ax.imshow(cobertura.values, aspect="auto", cmap=CMAP_SEQ, vmin=0, vmax=1,
          interpolation="nearest")

ax.set_yticks(range(len(cobertura.index)))
ax.set_yticklabels(cobertura.index, fontsize=8.5)
passo = max(1, len(cobertura.columns) // 22)
pos = range(0, len(cobertura.columns), passo)
ax.set_xticks(list(pos))
ax.set_xticklabels([pd.Timestamp(cobertura.columns[i]).strftime("%Y-%m") for i in pos],
                   rotation=45, ha="right", fontsize=8.5)
ax.grid(False)
for lado in ("top", "right", "left", "bottom"):
    ax.spines[lado].set_visible(False)

buracos = int((cobertura.values == 0).sum())
titular(ax, "Cobertura temporal por unidade da federação",
        f"Célula escura = há registro no período · Célula clara = lacuna · "
        f"{buracos:,} lacunas em {cobertura.size:,} células "
        f"({100*buracos/cobertura.size:.1f}%)".replace(",", "."))
ax.set_xlabel("Período")
plt.tight_layout()
plt.show()

In [ ]:
# ---- Gráfico 2: completude por coluna do arquivo original -----------
comp = (bruto.notna().mean() * 100).sort_values()

fig, ax = plt.subplots(figsize=(10, max(3, 0.42 * len(comp))))
ax.barh(range(len(comp)), comp.values, color=SERIES[0], height=0.62)
ax.set_yticks(range(len(comp)))
ax.set_yticklabels([str(c)[:34] for c in comp.index], fontsize=9.5)
ax.set_xlim(0, 108)
ax.set_xlabel("Preenchimento (%)")
for i, v in enumerate(comp.values):
    ax.text(v + 1.5, i, f"{v:.1f}%", va="center", fontsize=9, color=INK_2)
limpar(ax, "x")
titular(ax, "Completude das colunas do arquivo original",
        "Percentual de linhas com valor não nulo em cada coluna")
plt.tight_layout()
plt.show()

dups = bruto.duplicated().sum()
chave = [c for c in [col_data, col_uf, col_crime] if c is not None]
dups_chave = bruto.duplicated(subset=chave).sum() if chave else 0
print(f"Linhas totalmente duplicadas:                  {dups:,}")
print(f"Duplicatas na chave {chave}: {dups_chave:,}")
print(f"Registros com quantidade igual a zero:         {(df['qtd'] == 0).sum():,} "
      f"({100*(df['qtd'] == 0).mean():.1f}%)")
print(f"Registros com quantidade negativa:             {(df['qtd'] < 0).sum():,}")

In [ ]:
# ---- Gráfico 3: distribuição da variável observada -------------------
pos_vals = df.loc[df["qtd"] > 0, "qtd"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

ax = axes[0]
ax.hist(np.log10(pos_vals), bins=45, color=SERIES[0], edgecolor=SURFACE, linewidth=0.6)
ax.set_xlabel("log₁₀(nº de ocorrências)")
ax.set_ylabel("Frequência")
limpar(ax)
titular(ax, "Distribuição das ocorrências",
        "Escala log — a cauda longa é típica de contagens por UF")

ax = axes[1]
topo = df.groupby("crime")["qtd"].sum().nlargest(8).index
dados = [df.loc[df["crime"] == c, "qtd"].values for c in topo]
bp = ax.boxplot(dados, vert=False, patch_artist=True, widths=0.55,
                flierprops=dict(marker="o", markersize=2.5, alpha=0.35,
                                markerfacecolor=MUTED, markeredgecolor="none"))
for caixa in bp["boxes"]:
    caixa.set(facecolor=SERIES[0], alpha=0.75, edgecolor=SURFACE, linewidth=1.2)
for elem in ("whiskers", "caps", "medians"):
    for art in bp[elem]:
        art.set(color=INK_2, linewidth=1.2)
ax.set_yticks(range(1, len(topo) + 1))
ax.set_yticklabels([str(c)[:26] for c in topo], fontsize=9)
ax.set_xscale("log")
ax.set_xlabel("Nº de ocorrências por registro (escala log)")
limpar(ax, "x")
titular(ax, "Dispersão por tipo de crime",
        "Amplitude entre UFs dentro de cada categoria")

plt.tight_layout()
plt.show()

In [ ]:
# ---- Gráfico 4: composição por tipo de crime ------------------------
por_crime = df.groupby("crime")["qtd"].sum().sort_values()
part = 100 * por_crime / por_crime.sum()

fig, ax = plt.subplots(figsize=(10, max(3.2, 0.45 * len(por_crime))))
ax.barh(range(len(por_crime)), por_crime.values, color=SERIES[0], height=0.62)
ax.set_yticks(range(len(por_crime)))
ax.set_yticklabels([str(c)[:40] for c in por_crime.index], fontsize=9.5)
ax.xaxis.set_major_formatter(FMT_MILHAR)
ax.set_xlabel("Total de ocorrências no período")
folga = por_crime.max() * 0.02
for i, (v, p) in enumerate(zip(por_crime.values, part.values)):
    ax.text(v + folga, i, f"{p:.1f}%", va="center", fontsize=9, color=INK_2)
ax.set_xlim(0, por_crime.max() * 1.14)
limpar(ax, "x")
titular(ax, "Participação de cada tipo de crime no total",
        f"Acumulado de {df['data'].min():%Y-%m} a {df['data'].max():%Y-%m} · "
        f"rótulo = participação percentual")
plt.tight_layout()
plt.show()

In [ ]:
# ---- Gráfico 5: volume absoluto x taxa por 100 mil habitantes --------
# Duas medidas de escalas diferentes -> dois gráficos, nunca dois eixos y.
POP_2022 = {  # IBGE, Censo Demográfico 2022 (valores aproximados)
    "AC":830018,"AL":3127511,"AP":733759,"AM":3941613,"BA":14141626,"CE":8794957,
    "DF":2817381,"ES":3833712,"GO":7056495,"MA":6776699,"MT":3658649,"MS":2757013,
    "MG":20538718,"PA":8120131,"PB":3974687,"PR":11444380,"PE":9058931,"PI":3271199,
    "RJ":16054524,"RN":3302406,"RS":10882965,"RO":1581196,"RR":636707,"SC":7610361,
    "SP":44411238,"SE":2210004,"TO":1511460,
}

por_uf = df.groupby("uf")["qtd"].sum()
anos_eq = max(anos_cobertos, 1e-9)
taxa = (por_uf / pd.Series(POP_2022).reindex(por_uf.index) * 100_000 / anos_eq).dropna()

fig, axes = plt.subplots(1, 2, figsize=(13.5, 7.6))

ax = axes[0]
d = por_uf.sort_values()
ax.barh(range(len(d)), d.values, color=SERIES[0], height=0.68)
ax.set_yticks(range(len(d))); ax.set_yticklabels(d.index, fontsize=8.5)
ax.xaxis.set_major_formatter(FMT_MILHAR)
ax.set_xlabel("Total de ocorrências no período")
limpar(ax, "x")
titular(ax, "Volume absoluto por UF", "Domina o tamanho da população")

ax = axes[1]
d = taxa.sort_values()
ax.barh(range(len(d)), d.values, color=SERIES[1], height=0.68)
ax.set_yticks(range(len(d))); ax.set_yticklabels(d.index, fontsize=8.5)
ax.xaxis.set_major_formatter(FMT_MILHAR)
ax.set_xlabel("Ocorrências por 100 mil hab. por ano")
limpar(ax, "x")
titular(ax, "Taxa por 100 mil habitantes", "População IBGE/Censo 2022 — comparável entre UFs")

plt.tight_layout()
plt.show()

print("O contraste entre os dois painéis é o argumento central do Passo 2:")
print("priorizar pelo volume absoluto é priorizar pela população, não pelo risco.")

---
## 3.7 Exploração visual da série temporal

Esta é a exploração pedida no enunciado, nos moldes do exemplo da dívida líquida de Porto Alegre:
**série original**, **média móvel** para destacar a tendência e inspeção de **sazonalidade**.

In [ ]:
# Série agregada Brasil (soma de todas as UFs e tipos de crime)
serie = df.groupby("data")["qtd"].sum().sort_index()
serie.index = pd.DatetimeIndex(serie.index)

# Janela da média móvel = 1 ciclo anual completo na frequência detectada
JANELA = {"diária": 365, "semanal": 52, "mensal": 12, "trimestral": 4, "anual": 3}[FREQ]
mm = serie.rolling(JANELA, center=True, min_periods=max(2, JANELA // 2)).mean()

fig, ax = plt.subplots(figsize=(13.5, 5.6))
ax.plot(serie.index, serie.values, color=SERIES[0], linewidth=1.6,
        label="Série observada")
ax.plot(mm.index, mm.values, color=SERIES[1], linewidth=2.6,
        label=f"Média móvel ({JANELA} períodos)")

ax.yaxis.set_major_formatter(FMT_MILHAR)
ax.set_xlabel("Período")
ax.set_ylabel("Ocorrências")
ax.legend(loc="upper left", ncols=2)
limpar(ax)
titular(ax, "Total de ocorrências criminais no Brasil",
        f"Frequência {FREQ} · {df['data'].min():%Y-%m} a {df['data'].max():%Y-%m} · "
        f"soma de {n_ufs} UFs e {n_crimes} tipo(s) de crime")

# Rótulos diretos nos extremos, em vez de um número em cada ponto
for x, y, txt in [(serie.index[0], serie.iloc[0], "início"),
                  (serie.idxmax(), serie.max(), "pico"),
                  (serie.index[-1], serie.iloc[-1], "fim")]:
    ax.annotate(f"{txt}: {milhar(y)}", xy=(x, y), xytext=(0, 11),
                textcoords="offset points", ha="center", fontsize=9, color=INK_2)

plt.tight_layout()
plt.show()

var_total = 100 * (serie.iloc[-1] - serie.iloc[0]) / serie.iloc[0]
var_tend = 100 * (mm.dropna().iloc[-1] - mm.dropna().iloc[0]) / mm.dropna().iloc[0]
print(f"Variação ponta a ponta da série observada : {var_total:+.1f}%")
print(f"Variação ponta a ponta da média móvel     : {var_tend:+.1f}%  <- leitura de tendência")
print(f"Pico   : {serie.idxmax():%Y-%m}  ({milhar(serie.max())})")
print(f"Vale   : {serie.idxmin():%Y-%m}  ({milhar(serie.min())})")

In [ ]:
# ---- Séries por tipo de crime (small multiples) ---------------------
# Facetas: a identidade é carregada pelo título de cada painel, e não pela cor.
crimes_top = df.groupby("crime")["qtd"].sum().nlargest(9).index.tolist()
ncols = 3
nrows = int(np.ceil(len(crimes_top) / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.1 * nrows), sharex=True)
axes = np.atleast_1d(axes).ravel()

for ax, c in zip(axes, crimes_top):
    s = df[df["crime"] == c].groupby("data")["qtd"].sum().sort_index()
    m = s.rolling(JANELA, center=True, min_periods=max(2, JANELA // 2)).mean()
    ax.plot(s.index, s.values, color=SERIES[0], linewidth=1.3, alpha=0.55)
    ax.plot(m.index, m.values, color=SERIES[1], linewidth=2.2)
    ax.set_title(str(c)[:38], fontsize=10.5, loc="left", color=INK, pad=8)
    ax.yaxis.set_major_formatter(FMT_MILHAR)
    ax.tick_params(labelsize=8.5)
    limpar(ax)

for ax in axes[len(crimes_top):]:
    ax.set_visible(False)

fig.suptitle("Evolução por tipo de crime — escalas independentes", x=0.005, ha="left",
             fontsize=13, fontweight="semibold", color=INK, y=1.005)
fig.text(0.005, 0.985, "Linha clara = observado · linha laranja = média móvel de "
         f"{JANELA} períodos. Compare formatos, não níveis.",
         ha="left", fontsize=9.5, color=INK_2)
plt.tight_layout(rect=(0, 0, 1, 0.975))
plt.show()

In [ ]:
# ---- Séries das UFs de maior volume ---------------------------------
TOP_N = 6
ufs_top = df.groupby("uf")["qtd"].sum().nlargest(TOP_N).index.tolist()
cor_uf = {uf: SERIES[i] for i, uf in enumerate(ufs_top)}   # cor segue a entidade

fig, ax = plt.subplots(figsize=(13.5, 5.8))
for uf in ufs_top:
    s = df[df["uf"] == uf].groupby("data")["qtd"].sum().sort_index()
    ax.plot(s.index, s.values, color=cor_uf[uf], linewidth=2.0, label=uf)
    ax.annotate(uf, xy=(s.index[-1], s.iloc[-1]), xytext=(6, 0),
                textcoords="offset points", va="center", fontsize=10,
                color=cor_uf[uf], fontweight="semibold")

ax.yaxis.set_major_formatter(FMT_MILHAR)
ax.set_xlabel("Período")
ax.set_ylabel("Ocorrências")
ax.legend(loc="upper left", ncols=TOP_N)
ax.margins(x=0.04)
limpar(ax)
titular(ax, f"As {TOP_N} UFs de maior volume de ocorrências",
        "Séries rotuladas diretamente na ponta — a cor identifica a UF, não o ranking")
plt.tight_layout()
plt.show()

In [ ]:
# ---- Sazonalidade: perfil mensal e curvas ano a ano -----------------
if FREQ in ("mensal", "diária", "semanal"):
    mensal = df.groupby(["ano", "mes"])["qtd"].sum().reset_index()
    MESES = ["Jan","Fev","Mar","Abr","Mai","Jun","Jul","Ago","Set","Out","Nov","Dez"]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))

    # (a) dispersão do mês entre os anos
    ax = axes[0]
    dados = [mensal.loc[mensal["mes"] == m, "qtd"].values for m in range(1, 13)]
    bp = ax.boxplot(dados, patch_artist=True, widths=0.6,
                    flierprops=dict(marker="o", markersize=3, alpha=0.4,
                                    markerfacecolor=MUTED, markeredgecolor="none"))
    for caixa in bp["boxes"]:
        caixa.set(facecolor=SERIES[0], alpha=0.75, edgecolor=SURFACE, linewidth=1.2)
    for elem in ("whiskers", "caps", "medians"):
        for art in bp[elem]:
            art.set(color=INK_2, linewidth=1.2)
    media_geral = mensal["qtd"].mean()
    ax.axhline(media_geral, color=MUTED, linewidth=1.2, linestyle=(0, (4, 3)))
    ax.text(12.4, media_geral, "média", fontsize=9, color=MUTED, va="center")
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(MESES, fontsize=9)
    ax.yaxis.set_major_formatter(FMT_MILHAR)
    ax.set_ylabel("Ocorrências no mês")
    limpar(ax)
    titular(ax, "Perfil sazonal mensal",
            "Cada caixa reúne o mesmo mês de todos os anos da série")

    # (b) uma curva por ano — o ano é ordinal, então usa a rampa sequencial
    ax = axes[1]
    anos = sorted(mensal["ano"].unique())
    for i, a in enumerate(anos):
        s = mensal[mensal["ano"] == a].sort_values("mes")
        tom = CMAP_SEQ(0.25 + 0.7 * (i / max(1, len(anos) - 1)))
        ultimo = (i == len(anos) - 1)
        ax.plot(s["mes"], s["qtd"], color=tom,
                linewidth=2.6 if ultimo else 1.6, alpha=1.0 if ultimo else 0.85,
                marker="o", markersize=4.5 if ultimo else 0, label=str(a))
    ax.set_xticks(range(1, 13)); ax.set_xticklabels(MESES, fontsize=9)
    ax.yaxis.set_major_formatter(FMT_MILHAR)
    ax.set_ylabel("Ocorrências no mês")
    ax.legend(loc="best", ncols=min(4, len(anos)), fontsize=8.5, title="Ano",
              title_fontsize=8.5)
    limpar(ax)
    titular(ax, "Curvas sobrepostas ano a ano",
            "Tons mais escuros = anos mais recentes · curvas paralelas indicam sazonalidade estável")

    plt.tight_layout()
    plt.show()

    idx = mensal.groupby("mes")["qtd"].mean()
    idx = 100 * idx / idx.mean()
    print("Índice sazonal (100 = mês médio):")
    for m, v in idx.items():
        barra = "█" * int(abs(v - 100) * 1.2)
        print(f"  {MESES[m-1]}  {v:6.1f}  {barra}")
    print()
    print(f"Mês mais intenso : {MESES[int(idx.idxmax())-1]} ({idx.max():.1f})")
    print(f"Mês mais fraco   : {MESES[int(idx.idxmin())-1]} ({idx.min():.1f})")
    print(f"Amplitude sazonal: {idx.max() - idx.min():.1f} pontos percentuais")
else:
    print(f"Frequência {FREQ} — sem sazonalidade intra-anual observável nesta granularidade.")

In [ ]:
# ---- Heatmap ano x mês: magnitude -> rampa sequencial de um só matiz -
if FREQ in ("mensal", "diária", "semanal"):
    tabela = (df.pivot_table(index="ano", columns="mes", values="qtd", aggfunc="sum")
                .reindex(columns=range(1, 13)))

    fig, ax = plt.subplots(figsize=(12, max(3.2, 0.44 * len(tabela))))
    im = ax.imshow(tabela.values, aspect="auto", cmap=CMAP_SEQ, interpolation="nearest")

    ax.set_xticks(range(12)); ax.set_xticklabels(MESES, fontsize=9.5)
    ax.set_yticks(range(len(tabela.index)))
    ax.set_yticklabels(tabela.index, fontsize=9.5)
    ax.grid(False)
    for lado in ("top", "right", "left", "bottom"):
        ax.spines[lado].set_visible(False)

    cb = fig.colorbar(im, ax=ax, pad=0.015, fraction=0.03)
    cb.set_label("Ocorrências no mês", color=INK_2, fontsize=9.5)
    cb.ax.tick_params(labelsize=8.5, colors=MUTED)
    cb.outline.set_visible(False)

    titular(ax, "Ocorrências por ano e mês",
            "Leitura horizontal = sazonalidade · leitura vertical = tendência · "
            "célula branca = período ausente")
    plt.tight_layout()
    plt.show()

In [ ]:
# ---- Decomposição da série: tendência, sazonalidade e resíduo -------
try:
    from statsmodels.tsa.seasonal import STL

    s = serie.asfreq(FREQ_PANDAS)
    faltantes = int(s.isna().sum())
    if faltantes:
        print(f"{faltantes} período(s) sem observação — interpolados apenas para a decomposição.")
        s = s.interpolate("time")

    PERIODO = {"diária": 7, "semanal": 52, "mensal": 12, "trimestral": 4, "anual": 1}[FREQ]
    if PERIODO > 1 and len(s) >= 2 * PERIODO:
        res = STL(s, period=PERIODO, robust=True).fit()
        partes = [("Série observada", s, SERIES[0]),
                  ("Tendência", res.trend, SERIES[1]),
                  ("Sazonalidade", res.seasonal, SERIES[2]),
                  ("Resíduo", res.resid, MUTED)]

        fig, axes = plt.subplots(4, 1, figsize=(13.5, 10), sharex=True)
        for ax, (nome, dados, cor) in zip(axes, partes):
            if nome == "Resíduo":
                ax.axhline(0, color=AXIS, linewidth=1)
                ax.vlines(dados.index, 0, dados.values, color=cor, linewidth=1.1)
            else:
                ax.plot(dados.index, dados.values, color=cor, linewidth=2.0)
            ax.set_ylabel(nome, fontsize=10, color=INK_2)
            ax.yaxis.set_major_formatter(FMT_MILHAR)
            limpar(ax)
        axes[-1].set_xlabel("Período")
        fig.suptitle("Decomposição STL da série do Brasil", x=0.005, ha="left",
                     fontsize=13, fontweight="semibold", color=INK, y=0.998)
        fig.text(0.005, 0.978, "STL robusto — separa o movimento de longo prazo do "
                 "padrão que se repete a cada ano.", ha="left", fontsize=9.5, color=INK_2)
        plt.tight_layout(rect=(0, 0, 1, 0.97))
        plt.show()

        forca_tend = max(0, 1 - res.resid.var() / (res.trend + res.resid).var())
        forca_saz = max(0, 1 - res.resid.var() / (res.seasonal + res.resid).var())
        print(f"Força da tendência   : {forca_tend:.2f}  (0 = ausente, 1 = domina)")
        print(f"Força da sazonalidade: {forca_saz:.2f}")
        print()
        print("Referência usual: acima de 0,60 o componente é considerado forte.")
    else:
        print("Série curta demais para decomposição sazonal nesta frequência.")
except ImportError:
    print("statsmodels não instalado — execute: pip install statsmodels")

In [ ]:
# ---- Variação ano a ano: polaridade -> par divergente azul/vermelho --
anual = df.groupby("ano")["qtd"].sum()
yoy = (anual.pct_change() * 100).dropna()

fig, ax = plt.subplots(figsize=(12, 4.8))
cores = [POS if v > 0 else NEG for v in yoy.values]
ax.bar(yoy.index.astype(str), yoy.values, color=cores, width=0.6)
ax.axhline(0, color=AXIS, linewidth=1.2)
ax.set_ylabel("Variação sobre o ano anterior (%)")
folga = max(abs(yoy.values).max() * 0.06, 0.4)
for x, v in zip(yoy.index.astype(str), yoy.values):
    ax.text(x, v + (folga if v >= 0 else -folga), f"{v:+.1f}%",
            ha="center", va="bottom" if v >= 0 else "top", fontsize=9, color=INK_2)
ax.margins(y=0.22)
limpar(ax)

from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor=POS, label="Aumento"),
                   Patch(facecolor=NEG, label="Queda")], loc="best", ncols=2)
titular(ax, "Variação anual do total de ocorrências",
        "O indicador que as SSPs publicam hoje — sem separar tendência de sazonalidade")

if anual.index[-1] == df["data"].max().year and df["data"].max().month < 12 and FREQ == "mensal":
    ax.text(0.995, 0.02, f"⚠ {anual.index[-1]} incompleto", transform=ax.transAxes,
            ha="right", fontsize=9, color=INK_2, style="italic")

plt.tight_layout()
plt.show()

---
## 3.8 Resumo automático para o relatório

A célula abaixo consolida, com os números efetivamente observados, o texto dos itens 4 e 5 da
entrega. Basta revisar e colar no relatório.

In [ ]:
lacunas_pct = 100 * (cobertura.values == 0).sum() / cobertura.size
zeros_pct = 100 * (df["qtd"] == 0).mean()

linhas = [
    "=" * 78,
    "RESUMO PARA O RELATÓRIO — itens 4 e 5",
    "=" * 78,
    "",
    "4. DESCRIÇÃO DOS DADOS",
    f"   Variável observada : número de ocorrências criminais registradas",
    f"   Desagregação       : {n_ufs} unidades da federação x {n_crimes} tipo(s) de crime",
    f"   Frequência temporal: {FREQ}",
    f"   Período coberto    : {df['data'].min():%m/%Y} a {df['data'].max():%m/%Y} "
    f"({anos_cobertos:.1f} anos, {len(periodos)} períodos)",
    f"   Volume             : {milhar(len(df))} registros, "
    f"{milhar(df['qtd'].sum())} ocorrências no total",
    "",
    "   Completude / qualidade:",
    f"   - Grade UF x período preenchida em {100 - lacunas_pct:.1f}% das células "
    f"({milhar((cobertura.values == 0).sum())} lacunas)",
    f"   - {zeros_pct:.1f}% dos registros têm quantidade zero",
    f"   - {milhar(bruto.duplicated().sum())} linhas totalmente duplicadas no arquivo original",
    f"   - UFs ausentes: {', '.join(faltando) if faltando else 'nenhuma'}",
    f"   - {int(anos_cobertos)} ciclos anuais completos "
    f"({'suficiente' if int(anos_cobertos) >= 3 else 'pouco'} para estimar sazonalidade anual)",
    "",
    "5. EXPLORAÇÃO VISUAL INICIAL",
    f"   Nível inicial ({serie.index[0]:%m/%Y}) : {milhar(serie.iloc[0])}",
    f"   Nível final   ({serie.index[-1]:%m/%Y}) : {milhar(serie.iloc[-1])}",
    f"   Variação ponta a ponta                  : {var_total:+.1f}%",
    f"   Variação da média móvel de {JANELA} períodos : {var_tend:+.1f}%",
    f"   Pico da série : {serie.idxmax():%m/%Y} ({milhar(serie.max())})",
    f"   Vale da série : {serie.idxmin():%m/%Y} ({milhar(serie.min())})",
]

if FREQ in ("mensal", "diária", "semanal"):
    linhas += [
        f"   Mês sazonalmente mais intenso : {MESES[int(idx.idxmax())-1]} "
        f"(índice {idx.max():.1f})",
        f"   Mês sazonalmente mais fraco   : {MESES[int(idx.idxmin())-1]} "
        f"(índice {idx.min():.1f})",
        f"   Amplitude sazonal             : {idx.max() - idx.min():.1f} p.p.",
    ]

linhas += ["", "=" * 78]
for l in linhas:
    print(l)

---
## 3.9 Hipóteses iniciais formadas

> Preencher/confirmar conforme a saída das células acima. As afirmações abaixo são as hipóteses
> que a exploração visual permite levantar — **hipóteses, não conclusões**: nenhum teste formal
> foi aplicado, conforme o enunciado.

**Tendência.** A média móvel de um ciclo anual completo remove a oscilação intra-anual e deixa
visível o movimento de longo prazo. Compare a variação ponta a ponta da série observada com a da
média móvel (impressas na § 3.7): se as duas divergem muito, boa parte do que parece "tendência"
no gráfico bruto é, na verdade, o mês de início e o de fim caírem em pontos opostos do ciclo
sazonal — um erro de leitura comum nos boletins mensais.

**Sazonalidade.** Três evidências convergentes devem ser lidas em conjunto: (i) o índice sazonal
mensal da § 3.7 — quanto maior a amplitude entre o mês mais forte e o mais fraco, mais material é
o componente; (ii) as curvas ano a ano, que indicam sazonalidade **estável** quando são
aproximadamente paralelas e sazonalidade **instável** quando se cruzam; (iii) as faixas
horizontais recorrentes no heatmap ano × mês. Se a força sazonal da decomposição STL ficar acima
de 0,60, um modelo sem termo sazonal está descartado desde já.

**Quebras estruturais.** Observe especialmente 2020: a restrição de mobilidade da pandemia tende
a produzir uma queda abrupta nos crimes patrimoniais, que se comporta como *outlier de nível* e
não como ruído. Isso terá consequência direta na modelagem — será preciso tratar o período com
variável de intervenção, decomposição robusta ou recorte da amostra.

**Heterogeneidade entre UFs.** O contraste entre volume absoluto e taxa por 100 mil habitantes
(§ 3.6) mostra que as duas leituras produzem rankings diferentes. Para o projeto, isso sugere
tratar cada UF como uma série própria — ou adotar previsão hierárquica com reconciliação, em que
a soma das previsões estaduais é forçada a bater com a previsão do total nacional.

---

### Encaminhamento para as próximas etapas

| Etapa CRISP-DM | O que ficou definido aqui |
|---|---|
| **Preparação dos dados** | Tratar lacunas da grade UF × período; decidir entre imputação e recorte; padronizar a chave temporal; separar treino/teste respeitando a ordem cronológica |
| **Modelagem** | Série-alvo definida; frequência e período sazonal identificados; necessidade de termo sazonal e de tratamento do choque de 2020 já sinalizadas |
| **Avaliação** | Backtesting com origem móvel (*rolling origin*); baseline sazonal ingênuo como piso de comparação |
| **Comunicação** | O painel volume × taxa da § 3.6 e a decomposição da § 3.7 são as duas peças que traduzem o resultado para o gestor |